# Exploration des données: Give Me Some Credit

Analyse exploratoire du dataset GMSC avant nettoyage et imputation.


In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt

from src.data.load import load_raw

df = load_raw()
df.shape

## Vue d'ensemble

In [ ]:
print(f"Lignes : {df.shape[0]:,}")
print(f"Colonnes : {df.shape[1]}")
print(f"Taux de défaut : {df['SeriousDlqin2yrs'].mean() * 100:.2f} %")

In [ ]:
df.describe().T

## Répartition de la variable cible

In [ ]:
target_counts = df["SeriousDlqin2yrs"].value_counts().rename({0: "Pas de défaut", 1: "Défaut"})

fig = px.bar(
    x=target_counts.index,
    y=target_counts.values,
    labels={"x": "Statut", "y": "Nombre de clients"},
    text=target_counts.values,
    title="Répartition de la cible",
)
fig.show()


## Valeurs manquantes

In [ ]:
n_missing = df.isna().sum()
pct_missing = (n_missing / len(df) * 100).round(2)

report = (
    pd.DataFrame({"n_missing": n_missing, "pct_missing": pct_missing})
    .query("n_missing > 0")
    .sort_values("pct_missing", ascending=False)
)
display(report)

fig = px.bar(
    report,
    x=report.index,
    y="pct_missing",
    labels={"x": "Colonne", "pct_missing": "% manquant"},
    text="pct_missing",
    title="Pourcentage de valeurs manquantes par colonne",
)
fig.show()


In [ ]:
print(f"Nombre de lignes avec age = 0 : {(df['age'] == 0).sum()}")

fig_hist = px.histogram(df, x="age", nbins=50, title="Distribution de age")
fig_hist.show()

fig_box = px.box(df, y="age", title="Boxplot de age")
fig_box.show()

### Age = 0
Un âge de 0 est biologiquement impossible pour un emprunteur.

### Codes sentinelles 96/98

Les colonnes de retard de paiement contiennent des valeurs 96 et 98 qui apparaissent isolées, loin des valeurs normales (0-13). Ce sont des codes système, pas de vrais comptages.


In [ ]:
SENTINEL_COLUMNS = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
]

for col in SENTINEL_COLUMNS:
    n_sentinel = df[col].isin([96, 98]).sum()
    print(f"{col} : {n_sentinel} valeurs suspectes (96/98)")

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for i, col in enumerate(SENTINEL_COLUMNS):
    df[col].value_counts().sort_index().plot(kind="bar", ax=axes[i])
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xlabel("Valeur")
    axes[i].set_ylabel("Nombre de lignes")
plt.tight_layout()
plt.show()
